# Solutions — TPC-H Orders & Customers (Medium 05)

**Datasets:** `samples.tpch.orders`, `samples.tpch.customer`

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

orders   = spark.table("samples.tpch.orders")
customer = spark.table("samples.tpch.customer")

## Solution 1 — Total Spend & Order Count per Market Segment

In [ ]:
result_1 = (
    orders
    .join(customer, orders["o_custkey"] == customer["c_custkey"])
    .groupBy("c_mktsegment")
    .agg(
        F.round(F.sum("o_totalprice"), 2).alias("total_spend"),
        F.count("o_orderkey").alias("order_count")
    )
    .orderBy(F.col("total_spend").desc())
)

## Solution 2 — Top 5 Customers by Total Order Value

In [ ]:
result_2 = (
    orders
    .join(customer, orders["o_custkey"] == customer["c_custkey"])
    .groupBy("c_custkey","c_name","c_mktsegment")
    .agg(F.round(F.sum("o_totalprice"), 2).alias("total_order_value"))
    .orderBy(F.col("total_order_value").desc())
    .limit(5)
)

## Solution 3 — Avg/Min/Max Order Value per Market Segment

In [ ]:
result_3 = (
    orders
    .join(customer, orders["o_custkey"] == customer["c_custkey"])
    .groupBy("c_mktsegment")
    .agg(
        F.round(F.avg("o_totalprice"), 2).alias("avg_order_value"),
        F.round(F.min("o_totalprice"), 2).alias("min_order_value"),
        F.round(F.max("o_totalprice"), 2).alias("max_order_value")
    )
    .orderBy("c_mktsegment")
)

## Solution 4 — Customers with Orders at More Than 1 Priority Level

In [ ]:
result_4 = (
    orders
    .join(customer, orders["o_custkey"] == customer["c_custkey"])
    .groupBy("c_custkey","c_name")
    .agg(F.countDistinct("o_orderpriority").alias("priorities_used"))
    .filter(F.col("priorities_used") > 1)
    .orderBy(F.col("priorities_used").desc())
)

## Solution 5 — Rank Customers by Spend within Segment (Top 3)

In [ ]:
w = Window.partitionBy("c_mktsegment").orderBy(F.col("total_spend").desc())
result_5 = (
    orders
    .join(customer, orders["o_custkey"] == customer["c_custkey"])
    .groupBy("c_mktsegment","c_name")
    .agg(F.round(F.sum("o_totalprice"), 2).alias("total_spend"))
    .withColumn("rank_in_segment", F.rank().over(w))
    .filter(F.col("rank_in_segment") <= 3)
    .orderBy("c_mktsegment","rank_in_segment")
)

## Solution 6 — High-Priority Orders per Market Segment

In [ ]:
result_6 = (
    orders
    .filter(F.col("o_orderpriority").isin("1-URGENT","2-HIGH"))
    .join(customer, orders["o_custkey"] == customer["c_custkey"])
    .groupBy("c_mktsegment")
    .agg(
        F.count("o_orderkey").alias("high_priority_orders"),
        F.round(F.sum("o_totalprice"), 2).alias("high_priority_revenue")
    )
    .orderBy(F.col("high_priority_orders").desc())
)

## Solution 7 — Customers Who Never Placed an Order (Anti-Join)

In [ ]:
result_7 = (
    customer
    .join(orders.select("o_custkey").distinct(),
          customer["c_custkey"] == F.col("o_custkey"), how="left_anti")
    .select("c_custkey","c_name","c_mktsegment","c_acctbal")
)